In [4]:
!pip install streamlit openai pyngrok
!npm install -g localtunnel


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸
changed 22 packages in 3s
⠸
⠸3 packages are looking for funding
⠸  run `npm fund` for details
⠸

In [ ]:
%%writefile app.py
import streamlit as st
from openai import OpenAI

# --- 1. CONFIGURATION ---
st.title("Smart AI Assistant")

# Configure Client to point to Groq's Cloud Server
# REPLACE 'gsk_...' WITH YOUR ACTUAL API KEY
client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key="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" 
)

# --- 2. KNOWLEDGE BASE (The Facts) ---
knowledge_db = {
    "python": "Python is a high-level, interpreted programming language known for readability.",
    "streamlit": "Streamlit is an open-source framework used to build data web apps in pure Python.",
    "variable": "A variable is a symbolic name associated with a value and whose associated value may be changed.",
    "loop": "A loop is a sequence of instruction that is continually repeated until a certain condition is reached."
}

# --- 3. RETRIEVAL FUNCTION ---
def retrieve_context(query):
    query = query.lower()
    for key, value in knowledge_db.items():
        if key in query:
            # Return the exact text answer directly from the dictionary
            return value 
    
    # Return None if the keyword is not found
    return None 

# --- 4. GENERATION FUNCTION ---
def generate_answer(query, context):
    try:
        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile", # We use Llama 3 (very smart & fast)
            messages=[
                {
                    "role": "system", 
                    "content": (
                        "You are a helpful assistant."
                    )
                },
                {"role": "user", "content": f"Context: {context}\n\nQuestion: {query}"}
            ],
            temperature=0.5
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error connecting to Groq: {e}"

# --- 5. SESSION STATE & UI ---
# Initialize Chat History
if "chat_history" not in st.session_state:
    st.session_state.chat_history = {"Chat 1": []}
if "current_chat_id" not in st.session_state:
    st.session_state.current_chat_id = "Chat 1"

# Sidebar for History
st.sidebar.title("Chat History")
if st.sidebar.button("➕ New Chat"):
    new_id = f"Chat {len(st.session_state.chat_history) + 1}"
    st.session_state.chat_history[new_id] = []
    st.session_state.current_chat_id = new_id

for chat_id in st.session_state.chat_history:
    if st.sidebar.button(chat_id):
        st.session_state.current_chat_id = chat_id

# Main Chat Interface
active_id = st.session_state.current_chat_id
st.header(f"💬 {active_id}")

# Display History
for msg in st.session_state.chat_history[active_id]:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])

# Input Handling
if prompt := st.chat_input("Ask about Python concepts..."):
    # 1. User Message
    st.session_state.chat_history[active_id].append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.write(prompt)

    # 2. Retrieval & Generation
    # Check if the keyword exists in the knowledge_db
    context_match = retrieve_context(prompt)
    
    with st.chat_message("assistant"):
        if context_match:
            # CORE LOGIC: If there's a match in the DB, use the original answer directly without calling the LLM (Groq)
            answer = context_match
            st.write(answer)
        else:
            # If there's no match, show a spinner and call the LLM to process the query
            with st.spinner("Thinking..."):
                # Provide a general instruction to the LLM since there is no specific context from the DB
                answer = generate_answer(prompt, "Provide a concise and informative answer based on your general knowledge.")
                st.write(answer)
                
    # 3. Save the answer to the chat history
    st.session_state.chat_history[active_id].append({"role": "assistant", "content": answer})

Overwriting app.py


In [6]:
import subprocess
import time
import urllib.request

# 1. Run Streamlit in the background
subprocess.Popen(['streamlit', 'run', 'app.py'])
time.sleep(3)

# 2. Get the external IP (Localtunnel now requires this IP as a password)
print("🚀 Your application is almost ready!")
print("Copy the IP Address below to use as the 'Endpoint IP' / Password on the Localtunnel page.")
external_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
print(f"👉 Your IP Address: {external_ip}")

# 3. Run Localtunnel
!npx localtunnel --port 8501


  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://192.168.0.114:8501

🚀 Aplikasi Anda hampir siap!
PENTING: Copy IP Address di bawah ini. Anda akan membutuhkannya sebagai 'Endpoint IP' / Password di halaman Localtunnel.
👉 IP Address Anda: 182.253.92.135
⠙⠹⠸⠼⠴⠦⠧⠇your url is: https://light-foxes-press.loca.lt
  Stopping...


OSError: [Errno 5] Input/output error